# Notebook 2 — Financial Health Scoring Engine
**Purpose:** Develop, validate, and export the ML health scoring model.
Dimensions: Profitability (25%), Growth (20%), Leverage (20%), Cash Flow (15%), Dividend (10%), Trend (10%)

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns, sqlite3, os, sys
from sklearn.preprocessing import MinMaxScaler
sys.path.insert(0, os.path.abspath('..'))
sns.set_theme(style='darkgrid'); plt.rcParams['figure.figsize'] = (14, 6)

conn = sqlite3.connect(os.path.join('..', 'db.sqlite3'))
companies = pd.read_sql('SELECT * FROM dim_company', conn)
sectors = pd.read_sql('SELECT * FROM dim_sector', conn)
years = pd.read_sql('SELECT * FROM dim_year ORDER BY sort_order', conn)
pl = pd.read_sql('SELECT * FROM fact_profit_loss', conn).merge(years[['year_id','sort_order']], on='year_id')
bs = pd.read_sql('SELECT * FROM fact_balance_sheet', conn).merge(years[['year_id','sort_order']], on='year_id')
cf = pd.read_sql('SELECT * FROM fact_cash_flow', conn).merge(years[['year_id','sort_order']], on='year_id')
analysis = pd.read_sql('SELECT * FROM fact_analysis', conn)
companies = companies.merge(sectors[['sector_id','sector_name']], on='sector_id', how='left')
print(f'Loaded {len(companies)} companies')

## Step 1: Compute Metrics Per Company

In [ ]:
latest_pl = pl.sort_values('sort_order').groupby('company_id').last().reset_index()
latest_bs = bs.sort_values('sort_order').groupby('company_id').last().reset_index()
latest_cf = cf.sort_values('sort_order').groupby('company_id').last().reset_index()
growth_3y = analysis[analysis['period_label']=='3Y'][['company_id','compounded_sales_growth_pct']]

metrics = companies[['symbol','company_name','sector_name','roe_pct','roce_pct']].copy()
metrics = metrics.merge(latest_pl[['company_id','opm_pct','net_profit_margin_pct','interest_coverage','eps','dividend_payout_pct','net_profit','sales']], left_on='symbol', right_on='company_id', how='left')
metrics = metrics.merge(latest_bs[['company_id','debt_to_equity','equity_ratio']], left_on='symbol', right_on='company_id', how='left', suffixes=('','_bs'))
metrics = metrics.merge(latest_cf[['company_id','operating_activity','free_cash_flow']], left_on='symbol', right_on='company_id', how='left', suffixes=('','_cf'))
metrics = metrics.merge(growth_3y.rename(columns={'compounded_sales_growth_pct':'growth_3y'}), left_on='symbol', right_on='company_id', how='left', suffixes=('','_an'))

# Cash conversion ratio
metrics['cash_conversion'] = np.where(metrics['net_profit'] > 0, metrics['operating_activity'] / metrics['net_profit'], 0)
metrics.head()

## Step 2: Min-Max Normalization & Weighted Scoring

In [ ]:
def percentile_score(series, invert=False, cap=None):
    s = series.fillna(series.median())
    if cap: s = s.clip(upper=cap)
    if invert: s = -s
    return s.rank(pct=True) * 100

# Profitability (25%) — OPM + NPM + ROE
metrics['prof_score'] = (
    percentile_score(metrics['opm_pct']) * 0.4 +
    percentile_score(metrics['net_profit_margin_pct']) * 0.3 +
    percentile_score(metrics['roe_pct']) * 0.3
)

# Growth (20%) — 3Y Sales CAGR
metrics['grow_score'] = percentile_score(metrics['growth_3y'])
metrics.loc[metrics['growth_3y'] < 5, 'grow_score'] *= 0.7  # penalize < 5%

# Leverage (20%) — D/E inverse (lower = better)
metrics['lev_score'] = percentile_score(metrics['debt_to_equity'], invert=True, cap=5)
metrics.loc[metrics['debt_to_equity'] < 0.1, 'lev_score'] = 100  # debt-free = full

# Cash Flow (15%) — Cash conversion ratio
metrics['cf_score'] = percentile_score(metrics['cash_conversion'], cap=3)
metrics.loc[metrics['cash_conversion'] > 1.2, 'cf_score'] = metrics.loc[metrics['cash_conversion'] > 1.2, 'cf_score'].clip(lower=80)

# Dividend (10%) — payout consistency
metrics['div_score'] = percentile_score(metrics['dividend_payout_pct'])

# Trend (10%) — interest coverage + equity ratio
metrics['trend_score'] = (
    percentile_score(metrics['interest_coverage'], cap=20) * 0.5 +
    percentile_score(metrics['equity_ratio']) * 0.5
)

# Weighted overall
WEIGHTS = {'prof_score': 0.25, 'grow_score': 0.20, 'lev_score': 0.20,
           'cf_score': 0.15, 'div_score': 0.10, 'trend_score': 0.10}
metrics['overall_score'] = sum(metrics[k] * v for k, v in WEIGHTS.items())

# Labels
metrics['health_label'] = pd.cut(metrics['overall_score'],
    bins=[-1, 34, 49, 69, 84, 101],
    labels=['POOR','WEAK','AVERAGE','GOOD','EXCELLENT'])

print(metrics['health_label'].value_counts().sort_index(ascending=False))
metrics[['symbol','company_name','overall_score','health_label']].sort_values('overall_score', ascending=False).head(15)

## Step 3: Score Distribution Visualization

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
colors = {'EXCELLENT':'#10B981','GOOD':'#22C55E','AVERAGE':'#F59E0B','WEAK':'#F97316','POOR':'#EF4444'}
axes[0].hist(metrics['overall_score'], bins=20, color='#6366F1', edgecolor='white')
for thresh, label in [(85,'EXCELLENT'),(70,'GOOD'),(50,'AVERAGE'),(35,'WEAK')]:
    axes[0].axvline(x=thresh, color=colors.get(label,'gray'), linestyle='--', alpha=0.7, label=label)
axes[0].set_title('Health Score Distribution', fontweight='bold'); axes[0].legend()

label_counts = metrics['health_label'].value_counts()
axes[1].bar(label_counts.index, label_counts.values, color=[colors.get(l,'gray') for l in label_counts.index])
axes[1].set_title('Companies by Health Label', fontweight='bold')
plt.tight_layout(); plt.show()

## Step 4: Cross-Validation — Manual Check of 5 Companies

In [ ]:
check_symbols = ['TCS', 'HDFCBANK', 'WIPRO', 'ADANIPOWER', 'APOLLOHOSP']
check_cols = ['symbol','company_name','overall_score','health_label','prof_score','grow_score','lev_score','cf_score','div_score','trend_score']
validation = metrics[metrics['symbol'].isin(check_symbols)][check_cols].set_index('symbol')
print('=== Manual Validation ===')
for sym in check_symbols:
    if sym in validation.index:
        row = validation.loc[sym]
        print(f"\n{sym} ({row['company_name']})")
        print(f"  Overall: {row['overall_score']:.1f} → {row['health_label']}")
        print(f"  Prof={row['prof_score']:.0f} Growth={row['grow_score']:.0f} Lev={row['lev_score']:.0f} CF={row['cf_score']:.0f} Div={row['div_score']:.0f} Trend={row['trend_score']:.0f}")
    else:
        print(f"\n{sym}: Not found in dataset")

## Step 5: Sensitivity Analysis — Weight Changes

In [ ]:
# Test how changing each dimension's weight by ±5% affects the distribution
results = []
for dim in WEIGHTS:
    for delta in [-0.05, 0, 0.05]:
        w = WEIGHTS.copy()
        w[dim] += delta
        # Renormalize weights to sum to 1
        total_w = sum(w.values())
        w = {k: v/total_w for k, v in w.items()}
        scores = sum(metrics[k] * v for k, v in w.items())
        results.append({'dimension': dim, 'delta': delta, 'mean_score': scores.mean(), 'std_score': scores.std()})

sens_df = pd.DataFrame(results)
print(sens_df.to_string(index=False))

fig, ax = plt.subplots(figsize=(12, 5))
for dim in WEIGHTS:
    sub = sens_df[sens_df['dimension']==dim]
    ax.plot(sub['delta'], sub['mean_score'], marker='o', label=dim)
ax.set_xlabel('Weight Change'); ax.set_ylabel('Mean Overall Score')
ax.set_title('Sensitivity: Mean Score vs Weight Change per Dimension', fontweight='bold')
ax.legend(); plt.tight_layout(); plt.show()

## Step 6: Export Scores to CSV

In [ ]:
export_cols = ['symbol','company_name','sector_name','overall_score','health_label',
              'prof_score','grow_score','lev_score','cf_score','div_score','trend_score']
export_df = metrics[export_cols].round(2).sort_values('overall_score', ascending=False)
export_df.to_csv('../data/ml_health_scores.csv', index=False)
print(f'✅ Exported {len(export_df)} company scores to data/ml_health_scores.csv')
conn.close()